<p><font size="6" color='grey'><b>KI-Agenten. Planen. Handeln. Prüfen.</b></font></p>

<p><font size="5" color='grey'><b>Sessions, Checkpointing und Memory</b></font></p>

---

**Beitrag zum Leitprojekt:** Der PMO-Assistent kann eine Briefing-Sitzung fortsetzen, Zustände inspizieren und ausgewählte Präferenzen über Sitzungen hinweg wiederverwenden. Checkpointing und Memory bleiben dabei getrennte Schichten.

**Praxis-Transfer:** Mara Vogt arbeitet an einem Briefing zur Vektordatenbank-Entscheidung. Der laufende Dialog soll nach Unterbrechungen fortsetzbar bleiben; dauerhafte Präferenzen sollen nur kontrolliert gespeichert werden.

In [ ]:
#@title 🛠️ Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith-Variablen VOR allen LangChain-Imports setzen.
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m14_checkpointing"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(["OPENAI_API_KEY", "LANGSMITH_API_KEY"], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    JUDGE,
    PLANNER,
    WORKER,
    WORKER_PREMIUM,
    CODING,
    EMBEDDINGS,
)


# 1 | Übersicht
---



Checkpointing speichert den vollständigen Graph-State nach Verarbeitungsschritten. Eine `thread_id` verbindet mehrere Aufrufe zu einer Session. Memory ist eine separate Schicht für bewusst ausgewählte Informationen über Sitzungen hinweg.

| Lernschritt | Kernfrage | Prüfung |
|---|---|---|
| **Session** | Bleibt der laufende Dialog erhalten? | gleiche `thread_id` |
| **Isolation** | Vermischen sich Nutzer-Sessions? | unterschiedliche `thread_id` |
| **Persistenz** | Überlebt der State einen Neustart? | `SqliteSaver` |
| **Memory** | Was darf über Sessions hinweg erhalten bleiben? | Allowlist und Namespace |
| **Kontext** | Wie wird der Modellkontext begrenzt? | `trim_messages()` |

<p><font color='darkblue' size="4"><b>Viz</b></font></p>

- [Trimming und Summary](https://ralf-42.github.io/Agenten/visualisierungen/trimming_summary/)
- [Caching](https://ralf-42.github.io/Agenten/visualisierungen/caching/)

# 2 | Memory-Schichten: Was gehört wohin?
---



Checkpointing und Memory lösen unterschiedliche Probleme:

| Schicht | Inhalt | Lebensdauer | Beispiel |
|---|---|---|---|
| **Session-State** | laufender Workflow und Nachrichten | eine Session | `InMemorySaver`, `SqliteSaver` |
| **Conversation Buffer** | vollständiger Verlauf im aktuellen Thread | eine Session | alle Nachrichten weitergeben |
| **Sliding Window** | nur der jüngste Kontext | eine Session | `trim_messages()` |
| **Zusammenfassung** | komprimierter Verlauf | eine Session | alte Nachrichten verdichten |
| **Persistentes Memory** | bewusst freigegebene Präferenzen | über Sessions hinweg | Nutzer-Namespace oder Datenbank |

Buffer, Sliding Window und Zusammenfassung sind Strategien zur Kontextbegrenzung. Sie sind keine drei unabhängigen Langzeit-Memory-Systeme. Entity Memory und Vektorspeicher sind fortgeschrittene Varianten und werden in diesem Grundlagen-Notebook nur eingeordnet.

In [ ]:
#@markdown   <p><font size="4" color='green'>Memory-Schichten</font></p>
diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TB
    MEMORY[Memory] --> SESSION["Session-State\nCheckpointing"]
    MEMORY --> SHORT["Kurzzeit-Kontext"]
    MEMORY --> LONG["Persistentes Memory"]
    SHORT --> BUFFER["Conversation Buffer"]
    SHORT --> WINDOW["Sliding Window"]
    SHORT --> SUMMARY["Zusammenfassung"]
    LONG --> PREFS["Allowlist +\nPer-User-Namespace"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;

    %% Zuweisung
    class MEMORY input;
    class PREFS metric;
'''
mermaid(diagram, width=980)

# 3 | InMemorySaver: Session-Zustand fortsetzen
---



`InMemorySaver` speichert Checkpoints im Arbeitsspeicher. Für jede Session wird eine eigene `thread_id` verwendet. Der Graph bleibt bewusst einfach: Ein Node erhält den bisherigen Verlauf und erzeugt die nächste Antwort.

In [ ]:
from pathlib import Path
from typing import Annotated

from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, trim_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

In [ ]:
# LLM initialisieren
llm = init_chat_model(BASELINE)

In [ ]:
# State definieren
class SessionState(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# Nodes definieren
def briefing_assistant_node(state: SessionState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
# Graph definieren
session_builder = StateGraph(SessionState)

session_builder.add_node("briefing_assistant_node", briefing_assistant_node)

session_builder.add_edge(START, "briefing_assistant_node")
session_builder.add_edge("briefing_assistant_node", END)

session_memory = InMemorySaver()
session_agent = session_builder.compile(checkpointer=session_memory)

In [ ]:
# Viz Graph
from IPython.display import Image as IPImage, display

display(IPImage(session_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Anwendung 2 Threads Mara und Nora
thread_mara = {"configurable": {"thread_id": "mara-briefing"}}
thread_nora = {"configurable": {"thread_id": "nora-security"}}


def invoke_session(question: str, session_config: dict, run_name: str) -> str:
    result = session_agent.invoke(
        {"messages": [HumanMessage(content=question)]},
        config={
            **session_config,
            "run_name": run_name,
            "tags": ["inmemory-checkpointing"],
        },
    )
    return result["messages"][-1].content


invoke_session(
    "Merke dir: Thema ist die Vektordatenbank-Entscheidung. Kriterien: Belastungstest, Betriebsmodell, Kosten.",
    thread_mara,
    "mara_kontext",
)
mara_answer = invoke_session(
    "Welche Kriterien hatten wir für die Entscheidung festgelegt?",
    thread_mara,
    "mara_folgefrage",
)

invoke_session(
    "Merke dir: Thema ist Prompt-Injection und Tool-Grenzen.",
    thread_nora,
    "nora_kontext",
)
nora_answer = invoke_session(
    "Welche Kriterien hatten wir für die Entscheidung festgelegt?",
    thread_nora,
    "nora_folgefrage",
)

mprint(f"**Mara:** {mara_answer}")
mprint(f"**Nora:** {nora_answer}")

In [ ]:
#@markdown   <p><font size="4" color='green'>Thread-ID-Isolation</font></p>
diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart TD
    CP[("Checkpointer")]
    MARA["Mara\nthread_id = mara-briefing"] <-->|lesen / schreiben| CP
    NORA["Nora\nthread_id = nora-security"] <-->|lesen / schreiben| CP
    CP -->|"Mara: Vektordatenbank-Entscheidung"| M_STATE["Mara sieht nur ihre Session"]
    CP -->|"Nora: Prompt-Injection"| N_STATE["Nora sieht nur ihre Session"]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class MARA,NORA input;
    class M_STATE,N_STATE output;
    class CP target;
'''
mermaid(diagram, width=780)

# 4 | Thread-Isolation und State-Inspektion
---



Die gleiche `thread_id` verbindet Aufrufe. Unterschiedliche Werte isolieren die Zustände. `get_state()` liest den aktuellen Checkpoint; `get_state_history()` macht die Entwicklung der Session sichtbar.

In [ ]:
# Isolation Threads Mara und Nora
mara_state = session_agent.get_state(thread_mara)
nora_state = session_agent.get_state(thread_nora)

mara_human_messages = [
    message.content
    for message in mara_state.values.get("messages", [])
    if message.type == "human"
]
nora_human_messages = [
    message.content
    for message in nora_state.values.get("messages", [])
    if message.type == "human"
]

assert any("Belastungstest" in message for message in mara_human_messages)
assert any("Prompt-Injection" in message for message in nora_human_messages)
assert not any("Belastungstest" in message for message in nora_human_messages)
assert mara_state.values != nora_state.values

history = list(session_agent.get_state_history(thread_mara))
mprint(f"Mara: {len(mara_state.values.get('messages', []))} Nachrichten")
mprint(f"Nora: {len(nora_state.values.get('messages', []))} Nachrichten")
mprint(f"Mara: {len(history)} Checkpoints in der State-History")

In [ ]:
#@markdown   <p><font size="4" color='green'>Checkpoint-Verlauf</font></p>
diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    CP0[("Start\nleer")]
    CP1[("Checkpoint 1\nuser + AI")]
    CP2[("Checkpoint 2\nuser + AI\nuser + AI")]
    CP3[("Checkpoint 3\naktueller State")]
    CP0 --> CP1 --> CP2 --> CP3
    CURRENT["get_state\naktueller Checkpoint"] -.-> CP3
    HISTORY["get_state_history\nalle Checkpoints"] -.-> CP0

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class CP0,CP1,CP2,CP3 target;
'''
mermaid(diagram, width=900)

# 5 | SQLite: Persistenz über einen Neustart
---



`InMemorySaver` eignet sich für Tests. `SqliteSaver` schreibt die Checkpoints in eine lokale Datenbank. Zwei getrennte `with`-Blöcke simulieren, dass der Prozess zwischen den Aufrufen neu gestartet wurde.

Die Konfiguration bleibt bewusst in `sqlite_session_config`. Dadurch kann später keine wiederverwendete Variable aus einer anderen Demo versehentlich auf den falschen Thread zeigen.

In [ ]:
# Konstanten
DB_PATH = "./checkpoints.sqlite"
sqlite_session_config = {"configurable": {"thread_id": "sqlite-mara"}}

for suffix in ("", "-wal", "-shm"):
    database_file = Path(f"{DB_PATH}{suffix}")
    if database_file.exists():
        database_file.unlink()

In [ ]:
# DB aufbauen
with SqliteSaver.from_conn_string(DB_PATH) as saver:
    sqlite_agent = session_builder.compile(checkpointer=saver)
    sqlite_agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="Merke dir: Die Zielgruppe ist der Steuerkreis und braucht knappe Briefings."
                )
            ]
        },
        config={
            **sqlite_session_config,
            "run_name": "m14_sqlite_erstaufruf",
            "tags": ["sqlite"],
        },
    )

with SqliteSaver.from_conn_string(DB_PATH) as saver:
    sqlite_agent = session_builder.compile(checkpointer=saver)
    sqlite_result = sqlite_agent.invoke(
        {"messages": [HumanMessage(content="Was weißt du noch über die Zielgruppe?")]},
        config={
            **sqlite_session_config,
            "run_name": "m14_sqlite_neustart",
            "tags": ["sqlite"],
        },
    )
    sqlite_state = sqlite_agent.get_state(sqlite_session_config)


In [ ]:
# DB abrufen
sqlite_messages = sqlite_state.values.get("messages", [])
assert len(sqlite_messages) == 4
assert any(
    message.type == "human" and "knappe Briefings" in message.content
    for message in sqlite_messages
)
mprint(f"SQLite-Antwort nach Neustart: {sqlite_result['messages'][-1].content}")
mprint("✅ SQLite-Checkpoint überlebt den simulierten Prozessneustart.")

# 6 | Kontrolliertes Memory mit Nutzer-Isolation
---



Ein einfaches Memory benötigt drei Regeln: Nur freigegebene Inhalte werden geschrieben, sensible Inhalte werden abgewiesen, und jeder Abruf bleibt auf den Nutzer-Namespace begrenzt.

Eine Allowlist ist für eine Einführung verständlicher als eine lange Negativliste. In produktiven Systemen kommen zusätzlich Authentifizierung, Löschregeln und Zugriffskontrollen hinzu.

In [ ]:
briefing_memory_by_user: dict[str, dict[str, str]] = {}

In [ ]:
# Funktionen
def allowed_memory(text: str) -> bool:
    normalized = text.casefold()
    allowed_markers = (
        "präferenz",
        "ausgabeformat",
        "quellenhinweis",
        "out-of-corpus",
        "themeninteresse",
    )
    blocked_markers = (
        "api-key",
        "secret",
        "passwort",
        "rohnotiz",
        "privat",
        "nicht speichern",
    )
    return any(marker in normalized for marker in allowed_markers) and not any(
        marker in normalized for marker in blocked_markers
    )


def save_briefing_memory(user_id: str, key: str, text: str) -> str:
    if not allowed_memory(text):
        return "Nicht gespeichert: Information ist nicht für persistentes Memory freigegeben."
    briefing_memory_by_user.setdefault(user_id, {})[key] = text
    return f"Gespeichert für {user_id}: {text}"


def retrieve_briefing_memory(user_id: str, key: str) -> str:
    return briefing_memory_by_user.get(user_id, {}).get(
        key,
        "Keine passende Briefing-Präferenz gefunden.",
    )

In [ ]:
# Ausgabe persitenter Memory
mprint(save_briefing_memory("mara", "format", "Präferenz: Kurzfazit und Quellenhinweise."))
mprint(save_briefing_memory("max", "format", "Präferenz: Prompt-Injection und Tool-Grenzen prüfen."))
rejected_memory = save_briefing_memory(
    "mara",
    "notiz",
    "Nicht speichern: private Rohnotiz mit API-Key-Fragment.",
)
mprint(rejected_memory)

mara_preference = retrieve_briefing_memory("mara", "format")
max_preference = retrieve_briefing_memory("max", "format")
assert "Kurzfazit" in mara_preference
assert "Prompt-Injection" in max_preference
assert "Prompt-Injection" not in mara_preference
assert "Nicht gespeichert" in rejected_memory
assert retrieve_briefing_memory("mara", "notiz") == "Keine passende Briefing-Präferenz gefunden."
mprint("✅ Allowlist und Per-User-Isolation funktionieren.")

# 7 | Kurzzeit-Kontext begrenzen
---

Der Checkpoint speichert den vollständigen Verlauf aus dem Mara-Thread. Für den nächsten Modellaufruf wird daraus mit `trim_messages()` nur ein Ausschnitt weitergegeben. Der Checkpoint selbst bleibt unverändert.

Damit die Wirkung sichtbar ist, wird das Memory aus den vorherigen Kapiteln über `session_agent.get_state(thread_mara)` abgerufen und ausgegeben.




Ein Checkpoint kann den vollständigen Verlauf speichern, aber für den nächsten Modellaufruf ist manchmal nur ein Ausschnitt sinnvoll. `trim_messages()` begrenzt den weitergegebenen Kontext. Der Checkpoint selbst kann weiterhin mehr Nachrichten enthalten.

In [ ]:
def word_counter(messages: list) -> int:
    return sum(len(str(message.content).split()) for message in messages)


def display_role(message) -> str:
    return {
        "human": "Mensch",
        "ai": "KI",
        "system": "System",
    }.get(message.type, message.type)


mara_memory_state = session_agent.get_state(thread_mara)
mara_messages = list(mara_memory_state.values.get("messages", []))
assert mara_messages, "Der Mara-Thread enthält noch kein Memory."

trimmed_mara_messages = trim_messages(
    mara_messages,
    max_tokens=30,
    strategy="last",
    token_counter=word_counter,
    allow_partial=False,
)

mprint("### Vollständiger Verlauf aus dem Checkpoint")
for index, message in enumerate(mara_messages, 1):
    mprint(f"{index}. **{display_role(message)}:** {message.content}")

mprint("---")
mprint(
    f"Vollständiger Verlauf: {len(mara_messages)} Nachrichten; "
    f"weitergegebener Kontext: {len(trimmed_mara_messages)} Nachrichten."
)
mprint("### An das LLM weitergegebener Kontext")
for index, message in enumerate(trimmed_mara_messages, 1):
    mprint(f"{index}. **{display_role(message)}:** {message.content}")

assert len(trimmed_mara_messages) <= len(mara_messages)


# 8 | Kontext-Zusammenfassung
---
Der folgende Code verwendet denselben aus dem Checkpoint abgerufenen Mara-Verlauf wie Abschnitt 7. Die ältesten Nachrichten werden durch eine Summary ersetzt; die jüngsten Nachrichten bleiben erhalten.




Das Modul verbindet Session-Zustand, Checkpointing und kontrolliertes Memory in einer LangGraph-Anwendung.

<p><font color='black' size="5">
Kernaussagen
</font></p>

- `InMemorySaver` eignet sich für lokale Experimente und Tests; SQLite erhält Checkpoints über einen Neustart hinweg.
- Eine eindeutige `thread_id` trennt Gespräche und verhindert, dass Zustände verschiedener Nutzer vermischt werden.
- State-Inspektion macht gespeicherte Zustände und Checkpoint-Verläufe nachvollziehbar.
- `trim_messages()` aus Abschnitt 7 begrenzt den Kontext, ohne ältere Inhalte zu verdichten.
- Die folgende Strategie ergänzt das Trimming: Wenn `max_messages` überschritten wird, werden die ältesten Nachrichten zu einer Summary zusammengefasst; `RemoveMessage` entfernt sie aus dem State und die jüngsten `keep_recent` Nachrichten bleiben erhalten.
- Tests prüfen sowohl den fachlichen Ablauf als auch Isolation und Persistenz.
- Sicherheitsgrenzen bleiben Teil des Designs: keine unnötigen Daten speichern, Nutzerkontexte trennen und sensible Inhalte minimieren.

**Merksatz:** Memory ist kein unbeschränkter Gesprächsspeicher, sondern ein bewusst verwalteter Zustand mit klarer Lebensdauer, Isolation und Prüfbarkeit.


In [ ]:
from langchain_core.messages import RemoveMessage, SystemMessage

In [ ]:
# Funktionen
def summarize_old_messages(old_messages: list, *, summary_llm=None) -> str:
    """Fasst alte Nachrichten als Fakten und offene Fragen zusammen."""
    summary_llm = summary_llm or llm
    prompt = (
        "Fasse den folgenden Gesprächsverlauf knapp und sachlich zusammen. "
        "Erhalte wichtige Fakten, Entscheidungen und offene Fragen. "
        "Verwende nur Informationen, die im Verlauf belegt sind."
    )
    summary_response = summary_llm.invoke(
        [SystemMessage(content=prompt)] + old_messages
    )
    summary_content = summary_response.content
    return summary_content if isinstance(summary_content, str) else str(summary_content)


def build_summary_update(
    messages: list,
    *,
    max_messages: int = 6,
    keep_recent: int = 4,
    summary_llm=None,
) -> list:
    """Erzeugt RemoveMessage-Operationen plus eine Summary für den LangGraph-State."""
    if max_messages < 2:
        raise ValueError("max_messages muss mindestens 2 sein.")
    if not 1 <= keep_recent < max_messages:
        raise ValueError("keep_recent muss zwischen 1 und max_messages - 1 liegen.")
    if len(messages) <= max_messages:
        return []

    old_messages = messages[:-keep_recent]
    if any(message.id is None for message in old_messages):
        raise ValueError("Summary benötigt Nachrichten mit IDs aus dem LangGraph-State.")

    summary_text = summarize_old_messages(old_messages, summary_llm=summary_llm)
    summary_message = SystemMessage(
        content=f"Zusammenfassung bisher:\n{summary_text}"
    )
    remove_operations = [RemoveMessage(id=message.id) for message in old_messages]
    return remove_operations + [summary_message]

In [ ]:
max_messages = 3
keep_recent = 2
mara_summary_state = session_agent.get_state(thread_mara)
mara_messages_for_summary = list(
    mara_summary_state.values.get("messages", [])
)
assert mara_messages_for_summary, "Der Mara-Thread enthält noch kein Memory."

summary_update = build_summary_update(
    mara_messages_for_summary,
    max_messages=max_messages,
    keep_recent=keep_recent,
    summary_llm=llm,  # echte LLM-Zusammenfassung
)
compacted_state_messages = add_messages(
    mara_messages_for_summary,
    summary_update,
)

mprint("### Abgerufener Memory vor der Summary")
for index, message in enumerate(mara_messages_for_summary, 1):
    mprint(f"{index}. **{display_role(message)}:** {message.content}")

summary_messages = [
    message
    for message in compacted_state_messages
    if isinstance(message, SystemMessage)
]
recent_messages = [
    message
    for message in compacted_state_messages
    if not isinstance(message, SystemMessage)
]
summary_context = summary_messages + recent_messages

mprint("---")
mprint("### Vom LLM erzeugte Summary")
for message in summary_messages:
    mprint(message.content)

mprint(
    f"Summary ausgelöst bei mehr als {max_messages} Nachrichten; "
    f"entfernt: {len(summary_update) - 1}; "
    f"erhalten: {len(recent_messages)} jüngere Nachrichten."
)
mprint("### Summary und jüngere Nachrichten im komprimierten State")
for index, message in enumerate(compacted_state_messages, 1):
    mprint(f"{index}. **{display_role(message)}:** {message.content}")

mprint("### Kontext-Reihenfolge für das LLM")
for index, message in enumerate(summary_context, 1):
    mprint(f"{index}. **{display_role(message)}:** {message.content}")

# 9 | Sicherheitsgrenzen
---



| Risiko | Gegenmaßnahme |
|---|---|
| Fremder Zugriff auf eine Session | `thread_id` aus authentifiziertem Kontext ableiten |
| Vermischte Nutzerpräferenzen | Memory nach `user_id` oder einem gleichwertigen Namespace trennen |
| Secrets im Langzeit-Memory | Allowlist, Ablehnung sensibler Marker und Löschkonzept |
| Unbegrenzter Modellkontext | Verlauf vor dem Modellaufruf begrenzen oder zusammenfassen |

Eine `thread_id` ist kein Geheimnis und ersetzt keine Berechtigungsprüfung. Sie identifiziert einen State; der Zugriff darauf muss zusätzlich autorisiert werden.

In [ ]:
assert any("Belastungstest" in message for message in mara_human_messages)
assert not any("Belastungstest" in message for message in nora_human_messages)
assert any("knappe Briefings" in message.content for message in sqlite_messages)
assert "Kurzfazit" in mara_preference
assert "Prompt-Injection" not in mara_preference
mprint("✅ Alle Akzeptanzkriterien erfüllt.")

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

Code-Zellen enthalten bewusst Gerüste. Die Variablen und Selfchecks machen das erwartete Ergebnis sichtbar.




**Grundlagen**

Baue den InMemorySaver-Teil für einen eigenen Briefing-Fall um. Verwende zwei Aufrufe mit derselben `thread_id` und formuliere einen State-Selfcheck.

In [ ]:
# Grundlagen: eigene Session fortsetzen
grundlagen_ergebnis = None
# 1. Eigene Briefing-Frage mit session_agent und einer neuen thread_id beantworten.
# 2. Zweiten Aufruf mit derselben thread_id durchführen.
# 3. Den vollständigen State in grundlagen_ergebnis speichern.
# 4. Prüfen, dass der erste Nutzertext im State vorkommt.



**Aufbau**

Erweitere den Fall um einen zweiten Nutzer und SQLite. Zeige mit `get_state()` nach einem neu geöffneten Saver, dass der State erhalten bleibt und die Sessions getrennt bleiben.

In [ ]:
# Aufbau: SQLite und Session-Isolation
aufbau_ergebnisse = []
# 1. Eine zweite SQLite-Session mit eigener thread_id anlegen.
# 2. Den Saver in zwei getrennten with-Blöcken öffnen.
# 3. Beide States in aufbau_ergebnisse dokumentieren.
# 4. Prüfen, dass kein Kontext der anderen Session enthalten ist.



**Vertiefung**

Ergänze eine kontrollierte Memory-Regel für eine weitere Präferenz. Begründe, welche Informationen dauerhaft gespeichert werden dürfen und wie Löschung oder Ablauf umgesetzt würden.

In [ ]:
# Vertiefung: kontrolliertes Memory
vertiefung_ergebnis = None
# 1. Eine weitere Allowlist-Regel ergänzen.
# 2. Einen zweiten Nutzer mit eigener Präferenz speichern.
# 3. Den Abruf in vertiefung_ergebnis dokumentieren.
# 4. Eine abgewiesene sensible Information und die Nutzer-Isolation prüfen.

# B | Weiterführende Dokumentation
---



- [Checkpointing & Persistenz](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/checkpointing-persistenz.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Memory-Systeme](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/memory-systeme.html)
- [Human-in-the-Loop](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/human-in-the-loop.html)